In [3]:
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

In [4]:
df = pd.read_csv("housing.csv")

In [5]:
df.head(5)
df.dropna()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [6]:
X = df.drop(columns=["median_house_value", "ocean_proximity"])
y = pd.DataFrame({"median_house_value":df["median_house_value"]})

In [28]:
X.shape

(20640, 8)

In [30]:
y.head(5)

,median_house_value
0,452600.0
1,358500.0
2,352100.0
3,341300.0
4,342200.0


In [101]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

ValueError: Found input variables with inconsistent numbers of samples: [16512, 20640]

In [132]:

X = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_true = torch.tensor(y_train.to_numpy(), dtype=torch.float32)

X_mean = X.mean(dim=0, keepdim=True)
X_std = X.std(dim=0, keepdim=True)
X_scaled = (X - X_mean) / X_std
y_scaled = y_true / 100000.0

W = torch.randn(X.shape[1], 1, dtype=torch.float32, requires_grad=True)
b = torch.randn(1, 1, dtype=torch.float32, requires_grad=True)

epochs = 10000
rate = 0.01
for epoch in range(epochs):
    y_hat = X_scaled @ W + b
    loss = torch.mean((y_hat - y_scaled)**2)
    
    loss.backward()

    with torch.no_grad():
        W -= rate*W.grad; b -= rate*b.grad
    W.grad.zero_(); b.grad.zero_()

print(f"Final Loss: {loss.item()}")


Final Loss: 0.481113463640213


In [133]:
print(f"Actual Value: {y_true[0:5]}")
print(f"Predicted Value: {y_pred[0:5]}")

Actual Value: tensor([[103000.],
        [382100.],
        [172600.],
        [ 93400.],
        [ 96500.]])
Predicted Value: tensor([[181316.9844],
        [286467.5938],
        [263330.2188],
        [140990.1406],
        [177341.6562]], grad_fn=<SliceBackward0>)


In [151]:
X_t = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_t = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)
valid = ~torch.isnan(X_t).any(dim=1)
X_t = X_t[valid]
y_t = y_t[valid]
X_t_scaled = (X_t - X_mean) / X_std

with torch.no_grad():
    y_pred = (X_t_scaled @ W + b) * 100000.0
    mse = torch.mean((y_pred - y_t)**2)
    r_mse = torch.sqrt(mse)
    res = torch.sum((y_t - y_pred)**2)
    total = torch.sum((y_t - torch.mean(y_t))**2)
    r2 = 1 - (res / total)
    print(f"Test RMSE: ${r_mse.item():,.2f}")
    print(f"Test R^2 Score: {r2.item():.4f}")
    

Test RMSE: $70,384.03
Test R^2 Score: 0.6230
